# Project 13 — Local Multi-PDF Research Librarian
## Answer Questions Across Multiple Papers with Evidence

**Stack:** Ollama · LlamaIndex · Jupyter

In [ ]:
# !pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama

## Step 1 — Setup

In [ ]:
from llama_index.core import Settings, VectorStoreIndex, Document
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model="qwen3:8b", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

## Step 2 — Create Sample Research Papers

In [ ]:
papers = [
    Document(text="""Title: Attention Mechanisms in Neural Networks
Author: Smith et al., 2023

Abstract: This paper surveys attention mechanisms from basic additive attention
to multi-head self-attention used in transformers. We find that attention improves
model performance by 15-30% on sequence tasks while adding minimal parameters.

Key Findings:
1. Self-attention scales quadratically with sequence length O(n²)
2. Multi-head attention captures different types of relationships
3. Relative position encodings outperform absolute ones
4. Flash attention reduces memory from O(n²) to O(n)""",
    metadata={"paper_id": "P001", "year": 2023, "topic": "attention"}),

    Document(text="""Title: Efficient Retrieval-Augmented Generation
Author: Johnson et al., 2024

Abstract: We propose improvements to RAG pipelines that reduce latency by 40%
while maintaining answer quality. Key innovations include hierarchical retrieval,
context compression, and adaptive chunk sizing.

Key Findings:
1. Hierarchical retrieval (summary→detail) reduces retrievals by 60%
2. Context compression removes 40% of tokens with <2% quality loss
3. Adaptive chunks (200-800 tokens) outperform fixed-size chunks
4. Hybrid BM25+dense retrieval improves recall by 25%""",
    metadata={"paper_id": "P002", "year": 2024, "topic": "RAG"}),

    Document(text="""Title: Local Language Models for Enterprise
Author: Williams et al., 2024

Abstract: We evaluate running 7B-13B parameter models locally for enterprise
use cases. Results show local models achieve 85-92% of GPT-4 quality on
domain-specific tasks when combined with RAG and fine-tuning.

Key Findings:
1. 7B models with RAG match GPT-4 on closed-domain tasks
2. Fine-tuning on 1000 examples improves accuracy by 20%
3. Quantized models (4-bit) run at 30+ tokens/sec on consumer GPUs
4. Privacy-sensitive industries prefer local deployment 3:1""",
    metadata={"paper_id": "P003", "year": 2024, "topic": "local_llm"}),
]

print(f"Created {len(papers)} sample research papers")

## Step 3 — Build Cross-Paper Index

In [ ]:
index = VectorStoreIndex.from_documents(papers, show_progress=True)
query_engine = index.as_query_engine(similarity_top_k=3)
print("Cross-paper index built!")

## Step 4 — Cross-Reference Research Questions

In [ ]:
research_questions = [
    "What are the key findings about attention mechanism efficiency?",
    "How do local models compare to GPT-4 for enterprise use?",
    "What techniques reduce RAG latency?",
    "Compare the findings about retrieval improvements across papers.",
]

for q in research_questions:
    print(f"\n{'='*60}")
    print(f"Research Q: {q}")
    response = query_engine.query(q)
    print(f"\nAnswer: {response}")
    print(f"\nEvidence from:")
    for node in response.source_nodes:
        pid = node.metadata.get('paper_id', '?')
        score = node.score if hasattr(node, 'score') else 'N/A'
        print(f"  [{pid}] relevance={score}: {node.text[:80]}...")

## Step 5 — Research Synthesis

In [ ]:
from llama_index.core.query_engine import CitationQueryEngine

# Build a citation-aware query engine
citation_engine = CitationQueryEngine.from_args(index, similarity_top_k=3, citation_chunk_size=256)

synthesis_q = "Synthesize the key findings across all papers about improving AI system efficiency."
response = citation_engine.query(synthesis_q)
print("RESEARCH SYNTHESIS:")
print(response)
print("\nCitations:")
for i, node in enumerate(response.source_nodes):
    print(f"  [{i+1}] {node.text[:100]}...")

## What You Learned
- **Multi-document indexing** with paper metadata
- **Cross-reference queries** across papers
- **Citation-aware synthesis** combining evidence